# Experiment 3 — Designing a GNSS network for equatorial plasma bubbles

**Scientific question:** Where should a ground-based GNSS receiver network be placed, and how many receivers are useful for detecting and mapping nighttime equatorial ionospheric depletions over northeastern Brazil?

This self-contained OSSE uses hypothetical receiver sites and synthetic one-epoch satellite positions. It compares uniform, random, and information-optimized layouts on **held-out bubble realizations**. No station is claimed to be operating at a candidate location. The result is a planning demonstration, not an operational forecast.

**Prescribed truth → line-integrated observations → regularized retrieval.** PyIRI, if installed, supplies only the climatological electron-density background. It does not generate the disturbed plasma-bubble truth. Otherwise the notebook uses an explicitly labeled Chapman-like analytical background.

## Equations, units, and assumptions

The full state is reduced to a horizontal fractional depletion map $q(x,y)$ on a single thin-shell coordinate grid. The altitude dependence is prescribed separately:

$$N_e(x,y,z)=N_{e,0}(z)\,[1-q(x,y)\,h(z)], \qquad h(z)=\exp[-(z-z_b)^2/(2s_b^2)] .$$

Here $N_e$ is in m$^{-3}$, $z,z_b,s_b$ in km, and $0\leq q<1$. The synthetic bubble is an elongated, smooth depletion. This is a reduced-dimensional model: it resolves horizontal structure but prescribes the vertical depletion shape and assumes a horizontally uniform background profile.

For receiver position $\mathbf r_0$, satellite look unit vector $\hat{\mathbf l}$, and path length $s$ in m, the observation is a slant TEC line integral:

$$y_i=10^{-16}\int_{0}^{s_{top}} N_e(\mathbf r_0+s\hat{\mathbf l})\,ds + b_{r(i)}+b_{s(i)}+\epsilon_i \quad [\mathrm{TECU}],$$

where 1 TECU = $10^{16}$ electrons m$^{-2}$. The operator integrates through altitude and interpolates the horizontal depletion field along each ray. The mapping function used for the pierce-point visualization is **not** substituted for this line-integral observation operator.

Retrieval uses Gaussian-prior optimal estimation for the depletion grid. If $\mathbf d=\mathbf y_0-\mathbf y$ is the observed TEC deficit relative to the background, then $\hat q=q_a+S_aA^T(AS_aA^T+S_e)^{-1}(\mathbf d-Aq_a)$; the posterior covariance is $S=(S_a^{-1}+A^TS_e^{-1}A)^{-1}$. The prior, measurement noise, and model error are reported separately. Receiver/satellite biases can be injected; the baseline inversion does not silently know their values.

In [1]:
from pathlib import Path
import os, inspect, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve

SEED = 20261108
rng = np.random.default_rng(SEED)
ROOT = Path(os.environ.get("SBGEA_KEYNOTE_DATA", Path.home()/"Documents/Science/Programs/LFS/KEYNOTE")).expanduser()
DATA = ROOT/"outputs"/"data"/"osse"; FIG = ROOT/"outputs"/"figures"/"keynote"
DATA.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

CFG = {
    "center_lat_deg": -5.5, "center_lon_deg": -38.0,
    "grid_half_width_km": 750.0, "grid_n": 11,
    "candidate_count": 48, "receiver_counts": [3, 5, 10, 20, 40],
    "synthetic_satellites": 60, "max_rays_per_site": 8,
    "minimum_elevation_deg": 20.0, "ray_samples": 92,
    "observation_sigma_tecu": 0.20, "receiver_bias_sigma_tecu": 0.0,
    "satellite_bias_sigma_tecu": 0.0, "bubble_altitude_km": 350.0,
    "bubble_vertical_sigma_km": 100.0, "prior_sigma_fraction": 0.25,
    "prior_correlation_km": 230.0, "regularization_floor": 1e-5,
    "heldout_bubbles": 7, "null_trials": 5,
    "density_scale_error_test": 0.05,
}
LAT0, LON0 = CFG["center_lat_deg"], CFG["center_lon_deg"]
RE_KM = 6371.0; GNSS_ALT_KM = 20200.0
axis = np.linspace(-CFG["grid_half_width_km"], CFG["grid_half_width_km"], CFG["grid_n"])
GX, GY = np.meshgrid(axis, axis)
grid_xy = np.column_stack([GX.ravel(), GY.ravel()]); nstate = len(grid_xy)

# Hypothetical candidate envelope: inland points only in a coarse regional box.
# No road, terrain, land-tenure, power, permitting, or station-access data are implied.
cx = np.linspace(-850, 260, 8); cy = np.linspace(-700, 700, 6)
CAND_XY = np.array([(x,y) for y in cy for x in cx], dtype=float)
assert len(CAND_XY) == CFG["candidate_count"]

def xy_to_ll(x, y):
    lat = LAT0 + np.asarray(y)/111.0
    lon = LON0 + np.asarray(x)/(111.0*np.cos(np.deg2rad(LAT0)))
    return lat, lon

def ll_to_ecef(lat_deg, lon_deg, radius_km=RE_KM):
    lat, lon = np.deg2rad(lat_deg), np.deg2rad(lon_deg)
    return radius_km*np.array([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)])

def ecef_look_angles(site_xy, sat_xyz):
    lat, lon = xy_to_ll(*site_xy); lat, lon = np.deg2rad([float(lat), float(lon)])
    rec = ll_to_ecef(np.rad2deg(lat), np.rad2deg(lon))
    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([-np.sin(lat)*np.cos(lon), -np.sin(lat)*np.sin(lon), np.cos(lat)])
    up = np.array([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)])
    d = np.asarray(sat_xyz) - rec
    ee, nn, uu = d@east, d@north, d@up
    elev = np.rad2deg(np.arctan2(uu, np.hypot(ee,nn)))
    az = np.mod(np.rad2deg(np.arctan2(ee,nn)), 360.0)
    return az, elev

# Static synthetic constellation: global directions at GNSS orbital radius, not ephemerides.
rng_sat = np.random.default_rng(SEED+1)
unit = rng_sat.normal(size=(CFG["synthetic_satellites"],3)); unit /= np.linalg.norm(unit,axis=1)[:,None]
SAT_ECEF = (RE_KM+GNSS_ALT_KM)*unit

def site_rays(site_xy, min_elev=CFG["minimum_elevation_deg"]):
    az, el = ecef_look_angles(site_xy, SAT_ECEF)
    visible = np.flatnonzero(el >= min_elev)
    # Select the highest satellite in each 45-degree azimuth sector, then fill
    # remaining slots by elevation. This is a transparent sampling rule.
    chosen=[]
    for sector in range(8):
        ids=visible[(az[visible] >= sector*45)&(az[visible] < (sector+1)*45)]
        if len(ids): chosen.append(int(ids[np.argmax(el[ids])]))
    for j in visible[np.argsort(el[visible])[::-1]]:
        if len(chosen) >= CFG["max_rays_per_site"]: break
        if int(j) not in chosen: chosen.append(int(j))
    chosen=chosen[:CFG["max_rays_per_site"]]
    return [(float(az[j]),float(el[j]),int(j)) for j in chosen]

# Chapman-like fallback: approximate quiet nighttime F-region profile, normalized
# to a configurable peak. PyIRI is optional and only supplies this background.
ALT_KM = np.arange(90.0, 1000.1, 10.0)
PYIRI_AVAILABLE = False; BACKGROUND_SOURCE = "analytical Chapman-like fallback"
try:
    import PyIRI
    from PyIRI import sh_library as _pyiri_sh
    PYIRI_AVAILABLE = True
except ImportError:
    PyIRI = None; _pyiri_sh = None

def background_profile(alt_km):
    z=np.asarray(alt_km, dtype=float)
    if PYIRI_AVAILABLE:
        try:
            params=inspect.signature(_pyiri_sh.IRI_density_1day).parameters
            kwargs={"coeff_dir":None,"foF2_coeff":"URSI","hmF2_model":"SHU2015","coord":"GEO"}
            if "old_output" in params: kwargs["old_output"]=True
            elif "no_Es_old_output" in params: kwargs["no_Es_old_output"]=True
            result=_pyiri_sh.IRI_density_1day(2026,11,8,np.array([2.0]),np.array([LON0]),np.array([LAT0]),z,110.0,**kwargs)
            edp=np.asarray(result[-1])[0,:,0]
            if edp.shape == z.shape and np.all(np.isfinite(edp)) and np.nanmax(edp)>0:
                return edp, "PyIRI IRI_density_1day (climatological background)"
        except Exception as exc:
            warnings.warn(f"PyIRI present but call incompatible; using analytical fallback: {exc}")
    hm, scale, peak = 300.0, 55.0, 1.15e12
    zz=(z-hm)/scale
    chap=np.exp(0.5*(1.0-zz-np.exp(-zz)))
    chap=np.where((z>=100)&(z<=900),chap,0.0)
    return peak*chap, "analytical Chapman-like fallback"

NE0, BACKGROUND_SOURCE = background_profile(ALT_KM)
HBUB = np.exp(-0.5*((ALT_KM-CFG["bubble_altitude_km"])/CFG["bubble_vertical_sigma_km"])**2)
print("Background:", BACKGROUND_SOURCE, "| PyIRI available:", PYIRI_AVAILABLE)
print("Domain:", f"{xy_to_ll(-750,-750)[0]:.1f} to {xy_to_ll(750,750)[0]:.1f}° latitude,",
      f"{xy_to_ll(-750,-750)[1]:.1f} to {xy_to_ll(750,750)[1]:.1f}° longitude")
print("Hypothetical candidate sites:", len(CAND_XY), "| grid coefficients:", nstate)


Background: analytical Chapman-like fallback | PyIRI available: False
Domain: -12.3 to 1.3° latitude, -44.8 to -31.2° longitude
Hypothetical candidate sites: 48 | grid coefficients: 121


## Forward model and synthetic measurement design

The satellite directions are generated once as a **synthetic single-epoch constellation** on a spherical shell at 20,200 km altitude, then converted to receiver-local azimuth/elevation using ECEF-to-ENU geometry. These are not real ephemerides. Candidate receiver locations are a hypothetical inland grid around northeastern Brazil; accessibility is not verified and can be represented by editing the candidate mask/costs.

At each ray sample, the code integrates the background and the altitude-weighted depletion along the slant path. The state coefficients are interpolated bilinearly onto the horizontal grid. Synthetic observations add configurable white TEC noise and optional receiver/satellite biases. The default uses no injected biases; when enabled, these unmodeled offsets are reported as **model error**, not folded into the nominal measurement covariance.

In [2]:
def make_bubble(seed, empty=False):
    """Return a fractional depletion map and truth metadata; q is dimensionless."""
    rr=np.random.default_rng(seed)
    if empty: return np.zeros((CFG["grid_n"],CFG["grid_n"])), {"empty":True,"center":(np.nan,np.nan)}
    x0=rr.uniform(-180,180); y0=rr.uniform(-160,160)
    width=rr.uniform(65,115); length=rr.uniform(380,680); amp=rr.uniform(.42,.68)
    ang=rr.uniform(-18,18)*np.pi/180
    dx=GX-x0; dy=GY-y0
    cross=dx*np.cos(ang)-dy*np.sin(ang)
    along=dx*np.sin(ang)+dy*np.cos(ang)
    q=amp*np.exp(-.5*(cross/width)**2-.5*(along/(length/2.0))**2)
    return q, {"empty":False,"center":(x0,y0),"width_km":width,"length_km":length,"amplitude":amp,"angle_deg":np.rad2deg(ang)}

def bilinear_weights(x,y):
    x0,x1=axis[0],axis[-1]; dx=axis[1]-axis[0]
    fx=(x-x0)/dx; fy=(y-x0)/dx
    ix=np.floor(fx).astype(int); iy=np.floor(fy).astype(int)
    wx=fx-ix; wy=fy-iy
    return ix,iy,wx,wy

def ray_integral_row(site_xy, az_deg, elev_deg, density_scale=1.0):
    """Numerically integrate one ray; return baseline TECU and depletion row."""
    elev=np.deg2rad(elev_deg); az=np.deg2rad(az_deg)
    z=ALT_KM; dz_m=np.gradient(z)*1000.0
    path_factor=1.0/max(np.sin(elev),1e-6)
    along_km=z/np.tan(elev)
    xx=site_xy[0]+along_km*np.sin(az); yy=site_xy[1]+along_km*np.cos(az)
    base=float(np.sum(NE0*dz_m*path_factor)/1e16*density_scale)
    weights=NE0*HBUB*dz_m*path_factor/1e16*density_scale
    row=np.zeros(nstate)
    ix,iy,wx,wy=bilinear_weights(xx,yy)
    for ox,oy,wxy in ((0,0,(1-wx)*(1-wy)),(1,0,wx*(1-wy)),(0,1,(1-wx)*wy),(1,1,wx*wy)):
        ix1,iy1=ix+ox,iy+oy
        inside=(ix1>=0)&(ix1<CFG["grid_n"])&(iy1>=0)&(iy1<CFG["grid_n"])
        ids=iy1[inside]*CFG["grid_n"]+ix1[inside]
        np.add.at(row,ids,weights[inside]*wxy[inside])
    ipp_dist=CFG["bubble_altitude_km"]/np.tan(elev)
    ipp=(site_xy[0]+ipp_dist*np.sin(az),site_xy[1]+ipp_dist*np.cos(az))
    return base,row,ipp

def build_site_operator(site_xy, site_id, density_scale=1.0, ray_subset=None):
    rays=site_rays(site_xy)
    if ray_subset is not None: rays=[rays[j] for j in ray_subset]
    rows=[]; bases=[]; meta=[]; ipps=[]
    for az,el,satid in rays:
        b,a,ipp=ray_integral_row(site_xy,az,el,density_scale)
        rows.append(a); bases.append(b); ipps.append(ipp)
        meta.append({"site_id":int(site_id),"satellite_id":satid,"az_deg":az,"elev_deg":el,"ipp_x_km":ipp[0],"ipp_y_km":ipp[1]})
    return np.asarray(rows),np.asarray(bases),pd.DataFrame(meta),np.asarray(ipps)

# Cache the actual line-integral operator for every candidate station.
CANDIDATE_OPERATORS=[]
for idx,xy in enumerate(CAND_XY):
    a,b,m,ipp=build_site_operator(xy,idx)
    CANDIDATE_OPERATORS.append({"A":a,"base":b,"meta":m,"ipp":ipp})
ray_counts=np.array([len(item["base"]) for item in CANDIDATE_OPERATORS])
print("Visible ray count per candidate (min/median/max):", int(ray_counts.min()),float(np.median(ray_counts)),int(ray_counts.max()))
assert np.all(ray_counts>=4), "Synthetic constellation left a candidate with inadequate sky coverage"

# Gaussian smoothness prior on fractional depletion (not derived from truth).
delta=grid_xy[:,None,:]-grid_xy[None,:,:]
d2=np.sum(delta**2,axis=2)
SA=CFG["prior_sigma_fraction"]**2*np.exp(-0.5*d2/CFG["prior_correlation_km"]**2)
SA+=CFG["regularization_floor"]*np.eye(nstate)
PA=np.linalg.inv(SA); QPRIOR=np.full(nstate,0.02)
SIGMA=CFG["observation_sigma_tecu"]

def combine_sites(site_ids):
    As=[]; bs=[]; metas=[]; ipps=[]
    for sid in site_ids:
        op=CANDIDATE_OPERATORS[int(sid)]
        As.append(op["A"]); bs.append(op["base"]); metas.append(op["meta"]); ipps.append(op["ipp"])
    return np.vstack(As),np.concatenate(bs),pd.concat(metas,ignore_index=True),np.vstack(ipps)

def simulate_observations(A,base,q,meta,seed,measurement_sigma=SIGMA,receiver_bias_sigma=0.0,satellite_bias_sigma=0.0,density_scale_error=0.0):
    rr=np.random.default_rng(seed)
    truth_density_scale=1.0+density_scale_error
    # For the density-error experiment, callers provide an A/base built at the
    # perturbed scale; this argument documents the truth-side configuration.
    y=base-A@np.asarray(q).ravel()
    noise=rr.normal(0,measurement_sigma,len(y))
    rb={sid:rr.normal(0,receiver_bias_sigma) for sid in meta.site_id.unique()}
    sb={sid:rr.normal(0,satellite_bias_sigma) for sid in meta.satellite_id.unique()}
    bias=np.array([rb[s]+sb[t] for s,t in zip(meta.site_id,meta.satellite_id)])
    return y+noise+bias,{"noise":noise,"bias":bias,"truth_density_scale":truth_density_scale}

def retrieve(A,base,y,measurement_sigma=SIGMA):
    """Optimal-estimation posterior; no truth q is accepted by this function."""
    precision=PA+(A.T@A)/(measurement_sigma**2)
    cf=cho_factor(precision,lower=True,check_finite=False)
    post=cho_solve(cf,np.eye(nstate),check_finite=False)
    rhs=A.T@(base-y)/(measurement_sigma**2)
    qhat=QPRIOR+post@(rhs-(A.T@A@QPRIOR)/(measurement_sigma**2))
    post=(post+post.T)/2
    ak=post@(A.T@A)/(measurement_sigma**2)
    return qhat.reshape(CFG["grid_n"],CFG["grid_n"]),post,ak

def coverage_fraction(ipps,radius_km=220.0):
    dist=np.sqrt(((grid_xy[:,None,:]-ipps[None,:,:])**2).sum(axis=2))
    return float(np.mean(dist.min(axis=1)<radius_km))

def uniform_layout(n):
    """Greedy space-filling layout (geometry-only, independent of bubble truth)."""
    remaining=set(range(len(CAND_XY))); chosen=[int(np.argmin(np.sum(CAND_XY**2,axis=1)))]; remaining.remove(chosen[0])
    while len(chosen)<n:
        cand=np.array(sorted(remaining)); nearest=np.sqrt(((CAND_XY[cand,None,:]-CAND_XY[np.array(chosen)][None,:,:])**2).sum(axis=2)).min(axis=1)
        j=int(cand[np.argmax(nearest)]); chosen.append(j); remaining.remove(j)
    return chosen

def random_layout(n,seed): return np.random.default_rng(seed).choice(len(CAND_XY),size=n,replace=False).tolist()

def information_layout(max_n):
    """Greedy D-optimal siting using only candidate ray geometry and the prior."""
    cov=SA.copy(); remaining=set(range(len(CAND_XY))); selected=[]
    for _ in range(max_n):
        best=None; best_gain=-np.inf
        for sid in remaining:
            a=CANDIDATE_OPERATORS[sid]["A"]
            middle=np.eye(len(a))+(a@cov@a.T)/(SIGMA**2)
            sign,ld=np.linalg.slogdet(middle)
            gain=ld if sign>0 else -np.inf
            if gain>best_gain: best_gain=gain; best=sid
        a=CANDIDATE_OPERATORS[best]["A"]
        ac=a@cov
        cov=cov-ac.T@np.linalg.solve(SIGMA**2*np.eye(len(a))+ac@a.T,ac)
        cov=(cov+cov.T)/2
        selected.append(int(best)); remaining.remove(best)
    return selected


Visible ray count per candidate (min/median/max): 8 8.0 8


## Network-design OSSE and independent evaluation

Three layout strategies are compared at 3, 5, 10, 20, and 40 receivers:

- **Uniform:** greedy space-filling selection over candidate locations.
- **Random:** seeded random subsets.
- **Geometry-optimized:** greedy D-optimal selection that maximizes expected information under the stated prior and synthetic sky geometry. It never sees a bubble realization or its retrieval error.

Layouts are then evaluated on separate random bubble realizations and separate no-bubble trials. Metrics include full-map depletion RMSE (fraction), detection probability, false-alarm rate, mean posterior 1σ, a resolution-kernel proxy, pierce-point coverage, and ray count. These values describe this OSSE and its assumptions, not a forecast for a real Brazilian network.

In [3]:
def greedy_information_layout(max_n):
    return information_layout(max_n)

optimized_ids=greedy_information_layout(max(CFG["receiver_counts"]))
layout_cache={}; operator_cache={}
for n in CFG["receiver_counts"]:
    layout_cache[("uniform",n)]=uniform_layout(n)
    layout_cache[("random",n)]=random_layout(n,SEED+10*n)
    layout_cache[("geometry-optimized",n)]=optimized_ids[:n]

# Geometry-only check: clustered azimuths cannot resolve two horizontal components.
az_good=np.arange(0,360,45); az_bad=np.arange(0,24,4)
def los_condition(azimuths,elevation=45.0):
    a=np.deg2rad(azimuths); e=np.deg2rad(elevation)
    H=np.column_stack([np.cos(e)*np.sin(a),np.cos(e)*np.cos(a)])
    return float(np.linalg.cond(H.T@H))
condition_good=los_condition(az_good); condition_bad=los_condition(az_bad)
print(f"LOS geometry condition: distributed azimuths={condition_good:.2f}; clustered={condition_bad:.1f}")
assert condition_bad > condition_good*10

records=[]; representative={}
for strategy in ("uniform","random","geometry-optimized"):
    for n in CFG["receiver_counts"]:
        site_ids=layout_cache[(strategy,n)]
        A,base,meta,ipps=combine_sites(site_ids)
        # Keep the truth ensemble separate from the geometry-only layout optimization.
        errors=[]; detections=[]; null_false=[]
        for k in range(CFG["heldout_bubbles"]):
            qtruth,info=make_bubble(SEED+10000+100*k)
            y,parts=simulate_observations(A,base,qtruth,meta,SEED+20000+100*k)
            qhat,post,ak=retrieve(A,base,y)
            errors.append(np.sqrt(np.mean((qhat-qtruth)**2)))
            x0,y0=info["center"]; near=((GX-x0)**2+(GY-y0)**2)<(220.0**2)
            detections.append(bool(np.any(qhat[near]>.22)))
            if n==20 and strategy=="geometry-optimized" and k==0:
                representative={"A":A,"base":base,"meta":meta,"ipps":ipps,"qtruth":qtruth,"qhat":qhat,"post":post,"ak":ak,"y":y,"noise":parts["noise"],"info":info,"site_ids":site_ids}
        for k in range(CFG["null_trials"]):
            qzero=np.zeros((CFG["grid_n"],CFG["grid_n"]))
            y0,_=simulate_observations(A,base,qzero,meta,SEED+30000+100*k)
            qnull,_,_=retrieve(A,base,y0)
            null_false.append(bool(np.max(qnull)>.22))
        qhat,post,ak=retrieve(A,base,base)
        diag=np.clip(np.diag(post),0,None)
        resolution=float(np.mean(np.sqrt(np.diff(axis)[0]**2/np.maximum(np.diag(ak),1e-5))))
        records.append({"receivers":n,"strategy":strategy,"rmse_mean":float(np.mean(errors)),"rmse_std":float(np.std(errors,ddof=1)),
                        "detection_probability":float(np.mean(detections)),"false_alarm_rate":float(np.mean(null_false)),
                        "posterior_sigma_mean_fraction":float(np.mean(np.sqrt(diag))),"mean_resolution_km":resolution,
                        "coverage_fraction":coverage_fraction(ipps),"n_rays":len(base),"mean_echoes_per_receiver":len(base)/n})

sensitivity=pd.DataFrame(records)
sensitivity.to_csv(DATA/"experiment_3_network_sensitivity_reconstructed.csv",index=False)

# Density/model-error and optional bias stress tests at the nominal 20-site layout.
site_ids=layout_cache[("geometry-optimized",20)]
A0,b0,m0,ipp0=combine_sites(site_ids)
qstress,stress_info=make_bubble(SEED+777)
Apert,bpert,_,_=combine_sites(site_ids)  # base model used by the retrieval
# Truth density scaled by +5%, while retrieval retains the nominal density/operator.
ypert=(1+CFG["density_scale_error_test"])*(bpert-Apert@qstress.ravel())+np.random.default_rng(SEED+778).normal(0,SIGMA,len(bpert))
qpert,_,_=retrieve(Apert,bpert,ypert)
nominal_obs,_=simulate_observations(A0,b0,qstress,m0,SEED+778)
qnom,_,_=retrieve(A0,b0,nominal_obs)
model_error_rmse=float(np.sqrt(np.mean((qpert-qstress)**2)))
nominal_rmse=float(np.sqrt(np.mean((qnom-qstress)**2)))
print(f"20-site held-out density stress (+{100*CFG['density_scale_error_test']:.0f}%): RMSE {model_error_rmse:.3f}; nominal noisy case {nominal_rmse:.3f} (fractional depletion)")

# Consistency/validation checks from the generated results.
opt=sensitivity[sensitivity.strategy=="geometry-optimized"].sort_values("receivers")
assert np.all(np.diff(opt.posterior_sigma_mean_fraction)<=1e-7), "Adding geometry-optimized observations should not increase linear posterior uncertainty"
assert np.all(np.isfinite(sensitivity[["rmse_mean","posterior_sigma_mean_fraction","mean_resolution_km"]]))
assert sensitivity.n_rays.min()>0 and sensitivity.n_rays.max()<=CFG["synthetic_satellites"]*max(CFG["receiver_counts"])
assert model_error_rmse >= 0 and nominal_rmse >= 0
np.savez_compressed(DATA/"experiment_3_payload_reconstructed.npz",
    receiver_counts=np.array(CFG["receiver_counts"]),strategies=np.array(["uniform","random","geometry-optimized"]),
    rmse=np.array([[sensitivity[(sensitivity.strategy==s)&(sensitivity.receivers==n)].rmse_mean.iloc[0] for n in CFG["receiver_counts"]] for s in ["uniform","random","geometry-optimized"]]),
    rmse_std=np.array([[sensitivity[(sensitivity.strategy==s)&(sensitivity.receivers==n)].rmse_std.iloc[0] for n in CFG["receiver_counts"]] for s in ["uniform","random","geometry-optimized"]]),
    detect=np.array([sensitivity[(sensitivity.strategy=="geometry-optimized")&(sensitivity.receivers==n)].detection_probability.iloc[0] for n in CFG["receiver_counts"]]),
    false_alarm=np.array([sensitivity[(sensitivity.strategy=="geometry-optimized")&(sensitivity.receivers==n)].false_alarm_rate.iloc[0] for n in CFG["receiver_counts"]]),
    truth_map=representative["qtruth"],retrieval_map=representative["qhat"],candidate_xy=CAND_XY,
    selected_xy=CAND_XY[np.array(representative["site_ids"])],ipp_xy=representative["ipps"],grid_x=axis,grid_y=axis,
    detail_rmse=np.sqrt(np.mean((representative["qhat"]-representative["qtruth"])**2)),detail_ray_count=len(representative["base"]),
    detail_resolution_km=sensitivity[(sensitivity.strategy=="geometry-optimized")&(sensitivity.receivers==20)].mean_resolution_km.iloc[0],
    observation_tecu=representative["y"],background_tecu=representative["base"],noise_tecu=representative["noise"],
    posterior_sigma=np.sqrt(np.maximum(np.diag(representative["post"]),0)).reshape(CFG["grid_n"],CFG["grid_n"]),
    density_stress_rmse=model_error_rmse,nominal_rmse=nominal_rmse)
print("Network sensitivity rows:",len(sensitivity),"| held-out bubbles/layout:",CFG["heldout_bubbles"],"| null trials/layout:",CFG["null_trials"])
print(sensitivity[sensitivity.strategy=="geometry-optimized"][["receivers","n_rays","rmse_mean","detection_probability","false_alarm_rate","posterior_sigma_mean_fraction","coverage_fraction"]].to_string(index=False))


LOS geometry condition: distributed azimuths=1.00; clustered=69.9
20-site held-out density stress (+5%): RMSE 0.087; nominal noisy case 0.061 (fractional depletion)
Network sensitivity rows: 15 | held-out bubbles/layout: 7 | null trials/layout: 5
 receivers  n_rays  rmse_mean  detection_probability  false_alarm_rate  posterior_sigma_mean_fraction  coverage_fraction
         3      24   0.065113                    1.0               0.0                       0.118123           0.636364
         5      40   0.062173                    1.0               0.0                       0.079750           0.801653
        10      80   0.045013                    1.0               0.0                       0.042643           0.884298
        20     160   0.039385                    1.0               0.0                       0.029552           0.909091
        40     320   0.044974                    1.0               0.0                       0.026212           0.909091


## Truth, observations, and retrieval

The following figure is generated from the 20-receiver geometry-optimized case. Its layout was chosen from candidate-ray information alone; the displayed bubble realization was held out from that design step. The map is in local projected kilometers, with geographic secondary coordinates. The TEC panel shows actual synthetic slant observations against the background prediction. The posterior 1σ map is the local Gaussian covariance approximation from the regularized inverse problem.

In [4]:
# Keynote slide: actual held-out truth, posterior retrieval, GNSS geometry, and sensitivity.
from matplotlib.colors import TwoSlopeNorm
fig=plt.figure(figsize=(11.25,5.5),dpi=300,facecolor="white")
gs=fig.add_gridspec(2,3,left=.06,right=.975,top=.74,bottom=.285,wspace=.34,hspace=.42,width_ratios=[1,1,1.18])
ax0=fig.add_subplot(gs[:,0]); ax1=fig.add_subplot(gs[:,1]); ax2=fig.add_subplot(gs[0,2]); ax3=fig.add_subplot(gs[1,2])
fig.suptitle("REDE GNSS · Irregularidades equatoriais",x=.06,y=.965,ha="left",fontsize=16,fontweight="bold",color="#082F73")
fig.text(.06,.865,"Onde instalar receptores para observar bolhas de plasma?",fontsize=9.5,color="#25364A")
lat,lon=xy_to_ll(GX,GY)
lev=np.linspace(0,.7,15)
im0=ax0.contourf(lon,lat,representative["qtruth"],levels=lev,cmap="YlOrRd",extend="max")
ax0.contour(lon,lat,representative["qhat"],levels=[.2,.35],colors=["#082F73","#2686D0"],linewidths=[1.5,1.0])
site_xy=CAND_XY[np.array(representative["site_ids"])]
slat,slon=xy_to_ll(site_xy[:,0],site_xy[:,1]); ilat,ilon=xy_to_ll(representative["ipps"][:,0],representative["ipps"][:,1])
ax0.scatter(ilon,ilat,s=8,c="#2686D0",alpha=.28,label="pontos de perfuração")
ax0.scatter(slon,slat,s=24,marker="^",c="#082F73",edgecolors="white",linewidths=.4,label="receptores hipotéticos",zorder=5)
ax0.set(xlabel="Longitude (°)",ylabel="Latitude (°)"); ax0.tick_params(labelsize=6); ax0.xaxis.label.set_size(7); ax0.yaxis.label.set_size(7); ax0.grid(alpha=.17); ax0.legend(frameon=False,fontsize=6.5,loc="lower left")
ax0.set_title("Depleção verdadeira · contornos: recuperação",loc="left",fontsize=8,color="#082F73")
cb=fig.colorbar(im0,ax=ax0,fraction=.046,pad=.03); cb.ax.tick_params(labelsize=5.5); cb.ax.set_title("q",fontsize=6,pad=2)
im1=ax1.pcolormesh(axis,axis,representative["qhat"],shading="nearest",cmap="YlOrRd",vmin=0,vmax=.7)
ax1.contour(GX,GY,representative["qtruth"],levels=[.2,.4],colors=["#082F73","#2686D0"],linewidths=1.1)
ax1.set(xlabel="Leste (km)",ylabel="Norte (km)",aspect="equal"); ax1.tick_params(labelsize=6); ax1.xaxis.label.set_size(7); ax1.yaxis.label.set_size(7); ax1.grid(alpha=.15)
ax1.set_title("Recuperação · linhas: verdade",loc="left",fontsize=8,color="#082F73")
cb1=fig.colorbar(im1,ax=ax1,fraction=.046,pad=.03); cb1.ax.tick_params(labelsize=5.5); cb1.ax.set_title("q̂",fontsize=6,pad=2)
obs=representative["y"]; bkg=representative["base"]
ax2.plot(np.arange(len(obs)),bkg,"--",color="#7A8792",lw=1,label="fundo climatológico")
ax2.scatter(np.arange(len(obs)),obs,s=8,color="#EC8135",alpha=.72,label="TEC sintético")
ax2.set(xlabel="Raio observado",ylabel="TEC inclinada (TECU)"); ax2.tick_params(labelsize=5.5); ax2.xaxis.label.set_size(6.5); ax2.yaxis.label.set_size(6.5); ax2.grid(alpha=.18); ax2.legend(frameon=False,fontsize=5.8)
ax2.set_title("O instrumento observa TEC ao longo do raio",loc="left",fontsize=7.5,color="#082F73")
# Actual held-out validation RMSE, three strategies and five network sizes.
colors={"uniform":"#2686D0","random":"#70AE54","geometry-optimized":"#EC8135"}
for strategy,grp in sensitivity.groupby("strategy"):
 ax3.errorbar(grp.receivers,grp.rmse_mean,yerr=grp.rmse_std,marker="o",lw=1.3,capsize=2,color=colors[strategy],label=strategy.replace("geometry-optimized","otimizada").replace("uniform","uniforme").replace("random","aleatória"))
ax3.set(xlabel="Número de receptores",ylabel="RMSE da depleção (fração)",xticks=CFG["receiver_counts"]); ax3.tick_params(labelsize=5.5); ax3.xaxis.label.set_size(6.5); ax3.yaxis.label.set_size(6.5); ax3.grid(alpha=.18); ax3.legend(frameon=False,fontsize=5.8)
ax3.set_title("Avaliação em bolhas não usadas no projeto",loc="left",fontsize=7.5,color="#082F73")
row20=sensitivity[(sensitivity.strategy=="geometry-optimized")&(sensitivity.receivers==20)].iloc[0]
fig.text(.06,.175,f"20 receptores: {int(row20.n_rays)} raios · RMSE = {row20.rmse_mean:.2f} ± {row20.rmse_std:.2f} (fração) · detecção = {row20.detection_probability:.0%} · falso alarme = {row20.false_alarm_rate:.0%}.",fontsize=7.0,color="#25364A")
fig.text(.06,.105,"Rede hipotética; satélites sintéticos. A resolução é limitada pela geometria, ruído, prior e modelo vertical prescrito.",fontsize=6.5,color="#63758A")
fig.savefig(FIG/"Experiment_3_Slide.png",facecolor="white"); plt.show()

# A compact diagnostic: uncertainty/coverage versus network size.
figd,ax=plt.subplots(figsize=(7.2,4.2),dpi=180,facecolor="white")
for strategy,grp in sensitivity.groupby("strategy"):
 ax.plot(grp.receivers,grp.posterior_sigma_mean_fraction,"o-",color=colors[strategy],label=strategy)
ax.set(xlabel="Receptores",ylabel="Incerteza posterior média (fração)",title="Sensibilidade da incerteza à geometria da rede"); ax.grid(alpha=.2); ax.legend(frameon=False)
figd.tight_layout(); figd.savefig(FIG/"Experiment_3_Sensitivity_Reconstructed.png",facecolor="white"); plt.show()
print("Saved:",FIG/"Experiment_3_Slide.png")


Saved: $SBGEA_KEYNOTE_DATA/outputs/figures/keynote/Experiment_3_Slide.png


/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80549/3145211761.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(FIG/"Experiment_3_Slide.png",facecolor="white"); plt.show()
/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80549/3145211761.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  figd.tight_layout(); figd.savefig(FIG/"Experiment_3_Sensitivity_Reconstructed.png",facecolor="white"); plt.show()


## Three-experiment keynote composite

This final cell builds the 3375 × 1650 px keynote image from the **current numerical products** of all three notebooks. It reads the regenerated FPI and meteor-radar data products plus this GNSS run; no quantitative takeaway is hand-entered.

In [5]:
import textwrap
# Three aligned 3.75-inch columns, 11.25 x 5.5 in at 300 DPI.
fpi=np.load(DATA/"experiment_1_payload_reconstructed.npz")
rad=np.load(DATA/"experiment_2_payload_reconstructed.npz")
figc=plt.figure(figsize=(11.25,5.5),dpi=300,facecolor="white")
outer=figc.add_gridspec(1,3,left=.04,right=.985,top=.735,bottom=.17,wspace=.17)
navy="#082F73"; blue="#2686D0"; green="#70AE54"; orange="#EC8135"
# Column 1: FPI image and real sensitivity heatmap.
g0=outer[0,0].subgridspec(2,1,height_ratios=[1.05,.85],hspace=.58); a=figc.add_subplot(g0[0]); b=figc.add_subplot(g0[1])
a.imshow(fpi["image"],origin="lower",cmap="viridis"); a.set_title("Interferograma sintético",loc="left",fontsize=8,color=navy); a.axis("off")
fs=pd.read_csv(DATA/"experiment_1_sensitivity_reconstructed.csv")
piv=fs.pivot(index="fwhm_km",columns="qe",values="total_error_estimate_m_s")
b.imshow(piv.values,origin="lower",aspect="auto",cmap="magma",extent=[piv.columns.min()*100,piv.columns.max()*100,piv.index.min(),piv.index.max()]); b.set(xlabel="Eficiência quântica (%)",ylabel="FWHM camada (km)"); b.tick_params(labelsize=6); b.set_title("Erro total estimado (m/s)",loc="left",fontsize=8,color=navy)
# Column 2: sites and actual echo-count sensitivity.
g1=outer[0,1].subgridspec(2,1,height_ratios=[1.05,.85],hspace=.58); a=figc.add_subplot(g1[0]); b=figc.add_subplot(g1[1])
site_xy=np.asarray(rad["site_xy"]); meteor_xy=np.asarray(rad["meteor_xy"]).reshape(-1,2); a.scatter(meteor_xy[:,0],meteor_xy[:,1],s=2,color="#9DB4C5",alpha=.28); a.scatter(site_xy[:,0],site_xy[:,1],s=22,c=[navy]+[orange]*3,zorder=4); a.set(xlabel="Leste local (km)",ylabel="Norte local (km)"); a.xaxis.label.set_size(6.5); a.yaxis.label.set_size(6.5); a.grid(alpha=.15); a.set_title("Sítios e ecos simulados",loc="left",fontsize=7.2,color=navy); a.tick_params(labelsize=5.5)
es=pd.read_csv(DATA/"experiment_2_echo_sensitivity_reconstructed.csv")
b.errorbar(es.echoes,es.vector_rmse_m_s,yerr=es.formal_sigma_norm_m_s,fmt="o-",color=blue,capsize=2); b.set_xscale("log"); b.set(xlabel="Ecos utilizáveis",ylabel="Erro vetorial (m/s)"); b.xaxis.label.set_size(6.5); b.yaxis.label.set_size(6.5); b.grid(alpha=.15); b.tick_params(labelsize=5.5); b.set_title("Ruído e número de ecos",loc="left",fontsize=7.2,color=navy)
# Column 3: the current 20-receiver map plus network RMSE.
g2=outer[0,2].subgridspec(2,1,height_ratios=[1.05,.85],hspace=.58); a=figc.add_subplot(g2[0]); b=figc.add_subplot(g2[1])
a.pcolormesh(axis,axis,representative["qtruth"],shading="nearest",cmap="YlOrRd",vmin=0,vmax=.7); a.scatter(site_xy[:,0],site_xy[:,1],s=7,color=navy,marker="^"); a.set(xlabel="Leste (km)",ylabel="Norte (km)",aspect="equal"); a.xaxis.label.set_size(6.5); a.yaxis.label.set_size(6.5); a.tick_params(labelsize=5.5); a.grid(alpha=.12); a.set_title("Bolha prescrita + receptores",loc="left",fontsize=7.2,color=navy)
for strategy,grp in sensitivity.groupby("strategy"):
 b.plot(grp.receivers,grp.rmse_mean,"o-",lw=1,color={"uniform":blue,"random":green,"geometry-optimized":orange}[strategy],label=strategy.replace("geometry-optimized","otimizada").replace("uniform","uniforme").replace("random","aleatória"))
b.set(xlabel="Receptores GNSS",ylabel="RMSE (fração)",xticks=CFG["receiver_counts"]); b.xaxis.label.set_size(6.5); b.yaxis.label.set_size(6.5); b.grid(alpha=.15); b.tick_params(labelsize=5.5); b.legend(frameon=False,fontsize=5.4); b.set_title("Rede × erro de recuperação",loc="left",fontsize=7.2,color=navy)
heads=[("INTERFEROMETRIA","Como a eficiência do detector afeta a precisão dos ventos?"),("RADAR METEÓRICO","Quantas observações precisamos para medir o vento?"),("REDE GNSS","Onde instalar receptores para observar irregularidades?")]
for i,(title,question) in enumerate(heads):
 x=.04+i*.318
 figc.text(x,.94,title,color=navy,fontsize=10,fontweight="bold",ha="left")
 figc.text(x,.885,textwrap.fill(question,width=43),color="#25364A",fontsize=6.1,ha="left",va="top")
# Quantitative takeaways computed from the same tables/maps above.
frow=fs[(fs.qe==fs.qe.max())&(fs.fwhm_km==60)].iloc[0]
rrow=es.iloc[-1]
grow=sensitivity[(sensitivity.strategy=="geometry-optimized")&(sensitivity.receivers==20)].iloc[0]
figc.text(.035,.075,f"QE 95%, camada 60 km: erro estimado {frow.total_error_estimate_m_s:.1f} m/s",fontsize=6.3,color="#25364A")
figc.text(.365,.075,f"{int(rrow.echoes)} ecos: RMSE {rrow.vector_rmse_m_s:.1f} m/s",fontsize=6.3,color="#25364A")
figc.text(.685,.075,f"20 receptores: RMSE {grow.rmse_mean:.2f}; detecção {grow.detection_probability:.0%}",fontsize=6.3,color="#25364A")
figc.text(.035,.025,"Resultados de OSSEs didáticos · hipóteses e limitações descritas nos notebooks",fontsize=5.8,color="#63758A")
figc.savefig(FIG/"Three_Experiments_Keynote.png",dpi=300,facecolor="white"); plt.show()
from PIL import Image
imcheck=Image.open(FIG/"Three_Experiments_Keynote.png")
assert imcheck.size==(3375,1650), imcheck.size
print("Composite pixels:",imcheck.size,"| source data:",DATA)


Composite pixels: (3375, 1650) | source data: $SBGEA_KEYNOTE_DATA/outputs/data/osse


/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80549/2586230967.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  figc.savefig(FIG/"Three_Experiments_Keynote.png",dpi=300,facecolor="white"); plt.show()


## Validation, interpretation, and limitations

The notebook checks reproducibility, finite retrievals, and the expected reduction of linear posterior uncertainty as geometry-optimized sites are added. A deliberately clustered azimuth example verifies geometric ill-conditioning. The density stress test changes the forward truth by 5% while leaving the retrieval model at its nominal density; the resulting difference is model error, not measurement noise.

The simulated GNSS satellite positions are static shell points, not orbital propagation or a real almanac. The local tangent plane neglects Earth curvature along long rays; background density is horizontally uniform; the vertical bubble shape is prescribed; and the smoothness prior controls unresolved structure. Receiver bias treatment is an optional stress test and the nominal retrieval does not estimate biases. TEC tomography cannot resolve all small-scale scintillation or uniquely diagnose equatorial spread-F. Network access, site permissions, power, and actual GNSS visibility must be assessed before field deployment.

For an optional climatological background, the notebook uses the documented `PyIRI.sh_library.IRI_density_1day` interface when `PyIRI` is installed; the API is checked at runtime with `inspect.signature`. In this environment PyIRI is absent, so the Chapman-like fallback is used. PyIRI remains a background climatology and does not generate the disturbed bubble truth. References: [PyIRI documentation](https://pyiri.readthedocs.io/en/stable/), [PyIRI repository](https://github.com/victoriyaforsythe/PyIRI), [GNSS network design and TEC inversion context](https://doi.org/10.1002/navi.354).

### What could an undergraduate investigate next?

1. How does changing the satellite elevation mask affect map coverage and posterior uncertainty?
2. Can an explicit receiver/satellite bias state reduce false alarms without erasing a real bubble?
3. How do multiple bubble shapes and realistic satellite ephemerides change the network selected by the information-gain objective?

### Recovery provenance

This notebook is a **reconstructed, executable source**, built from the project specification and preserved OSSE payloads/sensitivity products. The original notebook bytes were deleted during migration; the original SHA-256 is retained in `KEYNOTE/MIGRATION_MANIFEST.csv` for audit but cannot be verified against the reconstructed source. Numerical outputs below are regenerated, not presented as a byte-identical recovery.

## Interactive GNSS experiment design

Select a receiver count/layout and a measurement-noise level. The cell regenerates TEC observations for the same held-out bubble, performs the optimal-estimation retrieval, and displays the retrieved map, truth contours, RMSE, and posterior uncertainty. It uses no truth information to choose the layout.

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
gnss_out=widgets.Output()
def _gnss_explore(receiver_count=20, strategy="geometry-optimized", sigma_tecu=0.20):
    with gnss_out:
        clear_output(wait=True)
        ids=layout_cache[(strategy,int(receiver_count))]
        A,b,meta,ipps=combine_sites(ids)
        qtruth=representative["qtruth"]
        y,_=simulate_observations(A,b,qtruth,meta,SEED+909,measurement_sigma=float(sigma_tecu))
        qhat,post,ak=retrieve(A,b,y,measurement_sigma=float(sigma_tecu))
        rmse=float(np.sqrt(np.mean((qhat-qtruth)**2)))
        sigma_map=np.sqrt(np.maximum(np.diag(post),0)).reshape(CFG["grid_n"],CFG["grid_n"])
        fig,ax=plt.subplots(1,2,figsize=(8,3.2),facecolor="white")
        im=ax[0].pcolormesh(axis,axis,qhat,shading="nearest",cmap="YlOrRd",vmin=0,vmax=.7)
        ax[0].contour(GX,GY,qtruth,levels=[.2,.4],colors=["#082F73","#2686D0"])
        ax[0].set(xlabel="Leste (km)",ylabel="Norte (km)",title=f"Recuperação · {receiver_count} receptores"); fig.colorbar(im,ax=ax[0],label="depleção")
        im2=ax[1].pcolormesh(axis,axis,sigma_map,shading="nearest",cmap="magma")
        ax[1].set(xlabel="Leste (km)",ylabel="Norte (km)",title="Incerteza posterior (1σ)"); fig.colorbar(im2,ax=ax[1],label="fração")
        fig.suptitle(f"RMSE={rmse:.3f} · {len(y)} raios TEC · σobs={sigma_tecu:.2f} TECU")
        fig.tight_layout(); plt.show()
        print("Posterior mean σ:",float(np.mean(sigma_map)),"fraction | IPP coverage:",f"{coverage_fraction(ipps):.0%}")
controls=widgets.interactive(_gnss_explore,
    receiver_count=widgets.SelectionSlider(options=CFG["receiver_counts"],value=20,description="Receptores"),
    strategy=widgets.Dropdown(options=["uniform","random","geometry-optimized"],value="geometry-optimized",description="Layout"),
    sigma_tecu=widgets.FloatSlider(value=SIGMA,min=.05,max=.80,step=.05,description="σ TECU"))
display(controls)


interactive(children=(SelectionSlider(description='Receptores', index=3, options=(3, 5, 10, 20, 40), value=20)…